<div style="font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif;max-width:900px;margin:0 auto;border-radius:16px;overflow:hidden;box-shadow:0 4px 20px rgba(0,0,0,0.12);border:1px solid #e2e2e2;">

<div style="background:linear-gradient(135deg,#002855 0%,#004b8d 55%,#0077c8 100%);padding:28px 20px 22px 20px;text-align:center;">
<img src="./img/ITESOLogo.png" alt="ITESO" width="260" style="margin-bottom:10px;">
<div style="color:#ffffff;font-size:15px;font-weight:600;letter-spacing:0.5px;text-transform:uppercase;opacity:0.9;">
Departamento de Electrónica, Sistemas e Informática
</div>
</div>

<div style="background-color:#ffffff;padding:26px 30px 30px 30px;text-align:center;">

<div style="color:#002855;font-size:26px;font-weight:800;margin-bottom:6px;">
Big Data Analysis
</div>

<div style="display:inline-block;background-color:#eaf4fb;color:#0077c8;font-size:13px;font-weight:700;padding:4px 14px;border-radius:20px;letter-spacing:0.5px;margin-bottom:22px;">
Autumn 2026
</div>


<hr style="border:none;border-top:2px solid #f0f0f0;margin:0 0 22px 0;">

<div style="background-color:#f7fafd;border-left:5px solid #0077c8;border-radius:8px;padding:14px 18px;text-align:left;margin-bottom:18px;">
<div style="font-size:12px;color:#7a7a7a;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px;">
Session 11
</div>
<div style="font-size:19px;color:#002855;font-weight:700;">
Batch Processing: JSON columns and Joins
</div>
</div>

<div style="font-size:14px;color:#444;margin-top:20px;">
<span style="font-weight:700;color:#002855;">Profesor:</span> Pablo Camarillo Ramírez
</div>

</div>
</div>

# Create SparkSession

In [28]:
from pcamarillor.spark_utils import SparkUtils

# Create Spark session
su = SparkUtils("Example - Extract info from JSON", "local[*]")

# Manipulating JSON Columns

In [32]:
json_schema = SparkUtils.generate_schema([("id", "int"), ("json_col", "string")])
json_data = [
    (1, '{"name": "Alice", "age": 25, "payments": [34, 433, 54], "address": {"city": "New York", "zip": "10001"}}'),
    (2, '{"name": "Bob", "age": 30, "address": {"city": "Los Angeles", "zip": "90001"}}'),
    (3, '{"name": "Charlie", "age": 35, "address": {"city": "Chicago", "zip": "60601"}}')
]

df_json = su._spark.createDataFrame(json_data, json_schema)
df_json.show(truncate=False)

+---+--------------------------------------------------------------------------------------------------------+
|id |json_col                                                                                                |
+---+--------------------------------------------------------------------------------------------------------+
|1  |{"name": "Alice", "age": 25, "payments": [34, 433, 54], "address": {"city": "New York", "zip": "10001"}}|
|2  |{"name": "Bob", "age": 30, "address": {"city": "Los Angeles", "zip": "90001"}}                          |
|3  |{"name": "Charlie", "age": 35, "address": {"city": "Chicago", "zip": "60601"}}                          |
+---+--------------------------------------------------------------------------------------------------------+



In [30]:
df_json_1 = df_json.select("json_col")

In [34]:
df_json_1.show(truncate=False)

+--------------------------------------------------------------------------------------------------------+
|json_col                                                                                                |
+--------------------------------------------------------------------------------------------------------+
|{"name": "Alice", "age": 25, "payments": [34, 433, 54], "address": {"city": "New York", "zip": "10001"}}|
|{"name": "Bob", "age": 30, "address": {"city": "Los Angeles", "zip": "90001"}}                          |
|{"name": "Charlie", "age": 35, "address": {"city": "Chicago", "zip": "60601"}}                          |
+--------------------------------------------------------------------------------------------------------+



### Extract a JSON column with get_json_object function

In [36]:
from pyspark.sql.functions import get_json_object
df_json = df_json.withColumn("name", get_json_object(df_json.json_col, "$.name"))
df_json.show(truncate=False)

+---+--------------------------------------------------------------------------------------------------------+-------+
|id |json_col                                                                                                |name   |
+---+--------------------------------------------------------------------------------------------------------+-------+
|1  |{"name": "Alice", "age": 25, "payments": [34, 433, 54], "address": {"city": "New York", "zip": "10001"}}|Alice  |
|2  |{"name": "Bob", "age": 30, "address": {"city": "Los Angeles", "zip": "90001"}}                          |Bob    |
|3  |{"name": "Charlie", "age": 35, "address": {"city": "Chicago", "zip": "60601"}}                          |Charlie|
+---+--------------------------------------------------------------------------------------------------------+-------+



In [37]:
df_json = df_json.withColumn("age", get_json_object(df_json.json_col, "$.age"))
df_json.show()

+---+--------------------+-------+---+
| id|            json_col|   name|age|
+---+--------------------+-------+---+
|  1|{"name": "Alice",...|  Alice| 25|
|  2|{"name": "Bob", "...|    Bob| 30|
|  3|{"name": "Charlie...|Charlie| 35|
+---+--------------------+-------+---+



In [38]:
df_json = df_json.withColumn("city", get_json_object(df_json.json_col, "$.address.city"))
df_json.show()

+---+--------------------+-------+---+-----------+
| id|            json_col|   name|age|       city|
+---+--------------------+-------+---+-----------+
|  1|{"name": "Alice",...|  Alice| 25|   New York|
|  2|{"name": "Bob", "...|    Bob| 30|Los Angeles|
|  3|{"name": "Charlie...|Charlie| 35|    Chicago|
+---+--------------------+-------+---+-----------+



In [39]:
df_json = df_json.withColumn("1st_payment", get_json_object(df_json.json_col, "$.payments[0]"))
df_json.show()

+---+--------------------+-------+---+-----------+-----------+
| id|            json_col|   name|age|       city|1st_payment|
+---+--------------------+-------+---+-----------+-----------+
|  1|{"name": "Alice",...|  Alice| 25|   New York|         34|
|  2|{"name": "Bob", "...|    Bob| 30|Los Angeles|       NULL|
|  3|{"name": "Charlie...|Charlie| 35|    Chicago|       NULL|
+---+--------------------+-------+---+-----------+-----------+



In [40]:
df_json = df_json.fillna({
    '1st_payment': 0
})
df_json.show()

+---+--------------------+-------+---+-----------+-----------+
| id|            json_col|   name|age|       city|1st_payment|
+---+--------------------+-------+---+-----------+-----------+
|  1|{"name": "Alice",...|  Alice| 25|   New York|         34|
|  2|{"name": "Bob", "...|    Bob| 30|Los Angeles|          0|
|  3|{"name": "Charlie...|Charlie| 35|    Chicago|          0|
+---+--------------------+-------+---+-----------+-----------+



### Extact a JSON column with from_json

In [41]:
from pyspark.sql.functions import from_json
# Deine the schema of the JSON object

columns_info = [
    ("name", "string"),
    ("age", "int"),
    ("payments", "array_int"),
    ("address", "struct", [
        ("city", "string"),
        ("zip", "string"),
    ]),
]

people_schema = SparkUtils.generate_schema(columns_info)

df_parsed = df_json.withColumn("parsed", from_json(df_json.json_col, people_schema))
df_parsed.printSchema()
df_parsed.show(truncate=False)

root
 |-- id: integer (nullable = true)
 |-- json_col: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- city: string (nullable = true)
 |-- 1st_payment: string (nullable = false)
 |-- parsed: struct (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- age: integer (nullable = true)
 |    |-- payments: array (nullable = true)
 |    |    |-- element: integer (containsNull = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- zip: string (nullable = true)

+---+--------------------------------------------------------------------------------------------------------+-------+---+-----------+-----------+---------------------------------------------+
|id |json_col                                                                                                |name   |age|city       |1st_payment|parsed                                       |
+---+------------------------

In [42]:
from pyspark.sql.functions import col
df_parsed.select(col("parsed.name"), col("1st_payment").alias("1st_payment_get_json"), col("parsed.payments").getItem(0).alias("1st_payment_from_json"), col("parsed.name")).show()

+-------+--------------------+---------------------+-------+
|   name|1st_payment_get_json|1st_payment_from_json|   name|
+-------+--------------------+---------------------+-------+
|  Alice|                  34|                   34|  Alice|
|    Bob|                   0|                 NULL|    Bob|
|Charlie|                   0|                 NULL|Charlie|
+-------+--------------------+---------------------+-------+



## Data Join

In [ ]:
df_a = su._spark.createDataFrame([(1, "Alice"), (2, "Bob")],
                              ["id", "name"])
df_b = su._spark.createDataFrame([(3, "Charlie"), (4, "David")],
                              ["id", "name"])
result = df_a.union(df_b)
result.show()

### Union w/o duplicates

In [ ]:
df_a = su._spark.createDataFrame([(1, "Alice"), (2, "Bob")],
                              ["id", "name"])
df_b = su._spark.createDataFrame([(1, "Alice"), (4, "David")],
                              ["id", "name"])
df_a.union(df_b).distinct().show()

### Union with Mismatched Schemas

In [ ]:
df_a = su._spark.createDataFrame([(1, "Alice")], ["id", "name"])
df_b = su._spark.createDataFrame([("Bob", 2)], ["name", "id"])
result = df_a.unionByName(df_b)
result.show()

### Union by Name with missing columns

In [ ]:
df_a = su._spark.createDataFrame([(1, "Alice", "NY")], ["id", "name", "city"])
df_a.show()
df_b = su._spark.createDataFrame([(2, "Bob")], ["id", "name"])
df_b.show()
result = df_a.unionByName(df_b, allowMissingColumns=True)
result.show()

## Left Join

### Datasets

In [43]:
book_data = [
    ("Game of thrones", 400, 1),
    ("Spark", 500, 2),
    ("Kafka", 300, 3),
    ("Java", 350, 5)
]
df_books = su._spark.createDataFrame(book_data, ["book_name", "cost", "writer_id"])

writer_data = [
    ("George R.R. Martin", 1),
    ("Zaharia", 2),
    ("Neha", 3),
    ("James", 4)
]
df_writers = su._spark.createDataFrame(writer_data, ["writer_name", "writer_id"])

df_books.show()
df_writers.show()

+---------------+----+---------+
|      book_name|cost|writer_id|
+---------------+----+---------+
|Game of thrones| 400|        1|
|          Spark| 500|        2|
|          Kafka| 300|        3|
|           Java| 350|        5|
+---------------+----+---------+

+------------------+---------+
|       writer_name|writer_id|
+------------------+---------+
|George R.R. Martin|        1|
|           Zaharia|        2|
|              Neha|        3|
|             James|        4|
+------------------+---------+



In [44]:
result = df_books.join(df_writers, 
      df_books["writer_id"] == df_writers["writer_id"], "left")
result.show()

+---------------+----+---------+------------------+---------+
|      book_name|cost|writer_id|       writer_name|writer_id|
+---------------+----+---------+------------------+---------+
|Game of thrones| 400|        1|George R.R. Martin|        1|
|          Spark| 500|        2|           Zaharia|        2|
|          Kafka| 300|        3|              Neha|        3|
|           Java| 350|        5|              NULL|     NULL|
+---------------+----+---------+------------------+---------+



In [45]:
result = df_books.join(df_writers, on="writer_id", how="left")
result.show()

+---------+---------------+----+------------------+
|writer_id|      book_name|cost|       writer_name|
+---------+---------------+----+------------------+
|        1|Game of thrones| 400|George R.R. Martin|
|        2|          Spark| 500|           Zaharia|
|        3|          Kafka| 300|              Neha|
|        5|           Java| 350|              NULL|
+---------+---------------+----+------------------+



## Right join

In [46]:
result = df_books.join(df_writers, df_books["writer_id"] == df_writers["writer_id"], "right")
result.show()

+---------------+----+---------+------------------+---------+
|      book_name|cost|writer_id|       writer_name|writer_id|
+---------------+----+---------+------------------+---------+
|Game of thrones| 400|        1|George R.R. Martin|        1|
|          Spark| 500|        2|           Zaharia|        2|
|          Kafka| 300|        3|              Neha|        3|
|           NULL|NULL|     NULL|             James|        4|
+---------------+----+---------+------------------+---------+



In [47]:
result = df_books.join(df_writers, on="writer_id", how="right")
result.show()

+---------+---------------+----+------------------+
|writer_id|      book_name|cost|       writer_name|
+---------+---------------+----+------------------+
|        1|Game of thrones| 400|George R.R. Martin|
|        2|          Spark| 500|           Zaharia|
|        3|          Kafka| 300|              Neha|
|        4|           NULL|NULL|             James|
+---------+---------------+----+------------------+



### Inner Join

In [48]:
df_books.join(df_writers, on="writer_id").show()

+---------+---------------+----+------------------+
|writer_id|      book_name|cost|       writer_name|
+---------+---------------+----+------------------+
|        1|Game of thrones| 400|George R.R. Martin|
|        2|          Spark| 500|           Zaharia|
|        3|          Kafka| 300|              Neha|
+---------+---------------+----+------------------+



In [49]:
su._spark.stop()